# 第 10 讲 · 模型训练二三事

本实验用几条 Alpaca 格式的本地样本和一个随机初始化的小型 Llama，
把第 9 讲的 tokenizer、模型和 LoRA 接到完整的一次训练上。
安装 `torch`、`datasets`、`transformers`、`tokenizers` 和 `peft` 后，可在 CPU 上离线运行。

## 1. 样本 → prompt / answer

把 `instruction` 和可选的 `input` 拼成 prompt，保留回答字段。过滤空回答后划分训练集与验证集。

In [ ]:
from datasets import Dataset

raw = Dataset.from_list([
    {"instruction": "What is 1 plus 1 ?", "input": "", "output": "It is 2 ."},
    {"instruction": "What is 2 plus 1 ?", "input": "", "output": "3 ."},
    {"instruction": "Translate hello .", "input": "to Chinese", "output": "你好 。"},
    {"instruction": "Say goodbye .", "input": "in English", "output": "Goodbye ."},
    {"instruction": "What is 2 plus 2 ?", "input": "", "output": "It is 4 ."},
    {"instruction": "What is 3 plus 2 ?", "input": "", "output": "5 ."},
    {"instruction": "This example has no answer .", "input": "", "output": ""},
])

def build_example(example):
    parts = (example["instruction"].strip(), example["input"].strip())
    return {
        "prompt": " ".join(part for part in parts if part),
        "answer": example["output"].strip(),
    }

examples = raw.filter(lambda row: bool(row["output"].strip()))
examples = examples.map(build_example, remove_columns=raw.column_names)
splits = examples.train_test_split(test_size=2, seed=10)
print(splits)
print(splits["train"][0])

## 2. 编码与回答区域的标签

本地词表只服务于这几条教学样本。每条序列依次是 `[USER]`、prompt、`[ASSISTANT]`、answer、`[EOS]`。
回答与 EOS 的标签保留 token ID，前面的标签设为 `-100`。输入与标签仍逐位置对齐。

In [ ]:
from tokenizers import Tokenizer
from tokenizers.models import WordLevel
from tokenizers.pre_tokenizers import WhitespaceSplit
from transformers import PreTrainedTokenizerFast

special = ["[PAD]", "[UNK]", "[USER]", "[ASSISTANT]", "[EOS]"]
words = {word for example in splits["train"] for field in ("prompt", "answer")
         for word in example[field].split()}
vocab = {word: i for i, word in enumerate(special + sorted(words))}
backend = Tokenizer(WordLevel(vocab=vocab, unk_token="[UNK]"))
backend.pre_tokenizer = WhitespaceSplit()
tokenizer = PreTrainedTokenizerFast(
    tokenizer_object=backend, pad_token="[PAD]", unk_token="[UNK]",
    eos_token="[EOS]", additional_special_tokens=["[USER]", "[ASSISTANT]"],
)

def encode_example(example):
    prompt_ids = tokenizer(
        f'[USER] {example["prompt"]} [ASSISTANT]', add_special_tokens=False,
    )["input_ids"]
    answer_ids = tokenizer(example["answer"], add_special_tokens=False)["input_ids"]
    answer_ids = answer_ids + [tokenizer.eos_token_id]
    input_ids = prompt_ids + answer_ids
    return {
        "input_ids": input_ids,
        "attention_mask": [1] * len(input_ids),
        "labels": [-100] * len(prompt_ids) + answer_ids,
    }

encoded = splits.map(encode_example, remove_columns=examples.column_names)
first = encoded["train"][0]
print(list(zip(tokenizer.convert_ids_to_tokens(first["input_ids"]), first["labels"])))

## 3. Collate 与 padding

`DataLoader` 收集变长样本，`collate` 在批内右侧填充。填充位置的 `attention_mask` 为 0，`labels` 为 `-100`。

In [ ]:
import torch
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader

def collate(rows):
    padding = {"input_ids": tokenizer.pad_token_id, "attention_mask": 0, "labels": -100}
    return {
        key: pad_sequence(
            [torch.tensor(row[key]) for row in rows],
            batch_first=True,
            padding_value=value,
        )
        for key, value in padding.items()
    }

train_loader = DataLoader(encoded["train"], batch_size=4, collate_fn=collate)
val_loader = DataLoader(encoded["test"], batch_size=2, collate_fn=collate)
batch = next(iter(train_loader))
for key, values in batch.items():
    print(key, tuple(values.shape), values.tolist())

padding = batch["attention_mask"] == 0
assert padding.any()
assert (batch["labels"][padding] == -100).all()

## 4. 手算 next-token loss

取 `logits[:, :-1]` 和 `labels[:, 1:]`。观察第一个回答 token 由它前一个位置的 logits 预测。
用 `assert_close` 验证手写的交叉熵与 Hugging Face 模型返回的 loss 一致。

In [ ]:
from torch.nn import functional as F
from transformers import LlamaConfig, LlamaForCausalLM

torch.manual_seed(10)
torch.set_num_threads(1)
config = LlamaConfig(
    vocab_size=len(tokenizer), hidden_size=32, intermediate_size=64,
    num_hidden_layers=1, num_attention_heads=4, num_key_value_heads=2,
    max_position_embeddings=64, pad_token_id=tokenizer.pad_token_id,
    eos_token_id=tokenizer.eos_token_id, use_cache=False,
)
base = LlamaForCausalLM(config)
with torch.no_grad():
    outputs = base(**batch)
    logits = outputs.logits[:, :-1, :].float()
    targets = batch["labels"][:, 1:]
    loss = F.cross_entropy(
        logits.reshape(-1, logits.shape[-1]), targets.reshape(-1), ignore_index=-100,
    )
torch.testing.assert_close(loss, outputs.loss)
print("监督 token 数", (targets != -100).sum().item(), "loss", loss.item())

## 5. 训练循环：只更新 LoRA 参数

在第 9 讲使用过的 `q_proj` 和 `v_proj` 上加 LoRA。观察训练前后 loss 与参数变化。
几条样本只用来检查计算流程。

In [ ]:
from peft import LoraConfig, TaskType, get_peft_model

model = get_peft_model(base, LoraConfig(
    task_type=TaskType.CAUSAL_LM, r=4, lora_alpha=8,
    target_modules=["q_proj", "v_proj"],
))
model.print_trainable_parameters()
trainable = [param for param in model.parameters() if param.requires_grad]
before = [param.detach().clone() for param in trainable]
optimizer = torch.optim.AdamW(trainable, lr=0.01)

model.train()
for step in range(4):
    for train_batch in train_loader:
        optimizer.zero_grad()
        loss = model(**train_batch).loss
        loss.backward()
        optimizer.step()
    print(f"step {step + 1}: train loss {loss.item():.4f}")

assert any(not torch.equal(old, new) for old, new in zip(before, trainable))
assert all(param.grad is None for param in model.parameters() if not param.requires_grad)

model.eval()
with torch.inference_mode():
    val_loss = model(**next(iter(val_loader))).loss
assert torch.isfinite(val_loss)
print(f"validation loss: {val_loss.item():.4f}")

**练习。** 把 prompt 的标签恢复成对应的输入 ID。比较监督 token 数和 loss，解释第一处发生变化的位置。